#### Mean Squared Error Loss

In [ ]:
import torch

def mse_loss(y_pred, y_true):
    return torch.mean((y_pred - y_true)**2)

y_pred = torch.randn(20)
y_true = torch.randn(20)

print(mse_loss(y_pred=y_pred, y_true=y_true))

# Compare with Pytorch implementation
loss = torch.nn.MSELoss()
print(loss(y_pred, y_true))

tensor(2.1121)
tensor(2.1121)


### Cross Entropy Loss

In [ ]:
import torch.nn as nn

def cross_entropy_loss(logits, targets, reduction='mean'):
    '''
    Args:
        logits: Tensor of shape (N, C) containing unnormalized operational scores
        targets: 1D Tensor of shape (N,) containing integer clas indices
        reduction: 'mean' | 'sum' | 'none'
    '''
    # 1. Log-Sum-Exp Trick for stable Softmax
    # Subtracting the max prevents float overflow from torch.exp()
    max_logits = torch.max(logits, dim=-1, keepdim=True)[0]
    shifted_logits = logits - max_logits

    # calculate the log of the denominator: log(sum(exp(x)))
    log_sum_exp = max_logits + torch.log(torch.sum(torch.exp(shifted_logits), dim=-1, keepdim=True)) # log(sigma(exp(x)))

    # 2. Gather the logits corresponding to the tru classes
    # Equivalent to log_softmax(logits) at the target indices
    N = logits.shape[0]
    row_indices = torch.arange(N, device=logits.device)
    target_logits = logits[row_indices, targets].unsqueeze(-1)

    # 3. Loss
    loss = -target_logits + log_sum_exp
    loss = loss.squeeze(-1) # Flatten back to (N,)

    # 4. Apply reduction
    if reduction == 'mean':
        return torch.mean(loss)
    elif reduction == 'sum':
        return torch.sum(loss)
    return loss


# For Soft Targets (Probability Distributions)
def cross_entropy_loss_soft(logits, targets, reduction='mean'):
    """
    Args: 
        logits: Tensor of shape (N, C)
        targets: Tensor of shape (N, C) containing true target probabilities
    """
    # Stable Log-Softmax implementation
    max_logits = torch.max(logits, dim=-1, keepdim=True)[0]
    log_softmax = (logits - max_logits) - torch.log(torch.sum(torch.exp(logits - max_logits), dim=-1, keepdim=True))

    # Cross-entropy calculation: -sum(target * log_softmax)
    loss = -torch.sum(targets * log_softmax, dim=-1)

    if reduction == 'mean':
        return torch.mean(loss)
    elif reduction == 'sum':
        return torch.sum(loss)
    return loss

# Test
logits = torch.tensor([[2.0, 0.5, 1.0], [0.1, 1.2, 2.1]], requires_grad=True) # (N=2, C=3)
targets = torch.tensor([0, 2]) # (N=2, )

# calculate cross entropy loss
loss_custom = cross_entropy_loss(logits, targets)
print(f"Custom Implementation: {loss_custom:.2f}")

# Compare with Pytorch loss
loss = nn.CrossEntropyLoss()
pytorch_loss = loss(logits, targets)
print(f"Pytorch implementation: {pytorch_loss:.2f}")

Custom Implementation: 0.45
Pytorch implementation: 0.45


### Binary Cross Entropy Loss

In [ ]:
def bce_loss_with_logits_loss(logits, targets, reduction='mean'):
    '''
        bce_loss = [y_n * log(x_n) + (1 - y_n) * log(1 - x_n)]
    Args:
        logits: Tensor of any shape containing unnormalized scores
        targets: Tensor of the same shape containing targets (0.0 or 1.0)
        reduction: 'mean' | 'sum' | 'none'
    '''

    # Numerically stable formation of BCE with logits
    # max(x, 0) - x * z + log(1 + exp(-|x|))
    # where x = logits, z = targets

    max_val = torch.clamp(logits, min=0)
    loss = max_val - logits * targets + torch.log(1.0 + torch.exp(-torch.abs(logits)))

    # apply reduction
    if reduction == 'mean':
        return torch.mean(loss)
    elif reduction == 'sum':
        return torch.sum(loss)
    return loss

# Standard BCE Loss (From Probabilities)
def custom_bce_loss(probs, targets, reduction='mean', eps=1e-12):
    """
    Args:
        probs: Tensor of probabilities (values strictly between 0 and 1)
        targets: Tensor of the same shape containing targets (0.0 or 1.0)
    """
    # Clamp to avoid log(0) -> -inf
    probs = torch.clamp(probs, min=eps, max=1.0 - eps)

    # Formula: -[z*log(p) + (1-z)*log(1-p)]
    loss = -(targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))

    if reduction == 'mean':
        return torch.mean(loss)
    elif reduction == 'sum':
        return torch.sum(loss)
    return loss